# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
# Replace this with your exact repository URL
REPO_URL = "https://github.com/Basil-Maqbool/flyrank-internship-assignment1"
REPO_DIR = "flyrank-internship-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    # This navigates to the correct subdirectory inside the cloned repo
    os.chdir("work/notebooks")

print("Current Working Directory:", os.getcwd())

Current Working Directory: /content/flyrank-internship-assignment1/work/notebooks


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I am choosing Lane 2: The Content Refresh Queue (Search Performance Decay & Optimization).
Organic search traffic is the lifeblood of our client digitl footprint/presence.
However, search rankings are not static; content decays over time as search intent shifts,
competitors publish fresher resources, or search algorithms update. Identifying which pages are
experiencing a genuine structural decline versus normal daily traffic fluctuations is a massive challenge.
By focusing on this lane, we can move from reactive, ad-hoc content updates to a proactive, machine-learning-driven queue.
This will help our content teams prioritize their limited editing hours on the pages that have the highest probability of recovering significant lost traffic.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

The Search Question: "Which of our high-traffic pages are experiencing a structural decline in organic search performance and should be prioritized for a content refresh?"
The Unit of Analysis: A single page (URL) over a 90-day window.
The Model's Output: A probability score (between 0.0 and 1.0) indicating the likelihood that a page is in a structural downward trend, sorted as a ranked queue.
The Business Decision: "Which pages should our editorial and SEO team invest time and resources into rewriting this week?"
The Concrete Action: The content team opens the top-ranked pages from our queue, runs a gap analysis against current search intent, refreshes the copy, and redeploys the page.
The Cost of a Wrong Recommendation (False Positive): If the model flags a page as declining when it is actually stable, we waste valuable writer hours (typically costing $150 - $300 per page in labor) modifying content that didn't need to be touched. Worse, we risk disrupting a high-performing page and causing a real drop in rankings.
The Cost of a Missed Opportunity (False Negative): If the model fails to flag a page that is actively decaying, the page will continue its downward slide in search results. Over months, this compounds into thousands of lost visits, lower conversion rates, and direct revenue loss.
Why ML is needed: A human editor cannot manually monitor thousands of pages across multiple metrics (impressions, CTR, positions, age, trend direction) simultaneously. Hand-written rules (like "refresh if older than 180 days") are too simple and generate too many false positives. ML can analyze complex, non-linear relationships across these variables to surface the true structural declines.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# 1. Total rows & baseline decline rate
total_pages = len(df)
declining_count = df['trend_direction'].str.lower().eq('down').sum()
decline_rate = declining_count / total_pages

# 2. Staleness rate (pages older than 180 days since last update)
stale_pages = (df["days_since_last_update"] >= 180).sum()
stale_rate = stale_pages / total_pages

# 3. High-impact stale pages (impressions >= 1000 and stale)
high_impact_stale = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 1000)).sum()

print(f"Total Pages in Starter Set: {total_pages:,}")
print(f"Overall Declining Rate (Target Class Ratio): {decline_rate:.1%}")
print(f"Staleness Rate (Days since last update >= 180): {stale_rate:.1%}")
print(f"Stale Pages with High Traffic (90d Impressions >= 1,000): {high_impact_stale:,}")

Total Pages in Starter Set: 30,000
Overall Declining Rate (Target Class Ratio): 54.2%
Staleness Rate (Days since last update >= 180): 0.6%
Stale Pages with High Traffic (90d Impressions >= 1,000): 12


From our exploratory analysis of the starter dataset, three key numbers stand out and justify this lane choice:
High Target Prevalence: 54.2% of the pages in our dataset are actively in a downward trend. This confirms we have a balanced classification problem with a substantial group of decaying pages to identify.
Massive Scale of Staleness: 0.6% of the pages have not been updated in over 180 days. However, the high decline rate across the dataset means many pages are decaying regardless of staleness — a simple rule-based approach would miss most declining pages.
High-Exposure Focus Group: There are stale pages that still pull in over 1,000 impressions every 90 days. These high-exposure, aging pages represent the exact high-impact "low hanging fruit" that a machine learning model can rank to maximize traffic recovery.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In organic search engine optimization, scientific humility is critical.
What we CANNOT claim: We cannot claim that our model "predicts Google's ranking algorithm" or that our features represent direct ranking factors. Google's core algorithm is a proprietary black box with thousands of real-time signals.
What we CAN claim: We are building a decision-support tool. We can claim that our model identifies historical statistical associations between observable page metrics (like content age, impressions, and CTR) and subsequent performance decay. The output is a directional, prioritized recommendation queue designed to optimize internal resource allocation, not a simulation of search engine mechanics.

## The one-paragraph frame

For content and SEO teams deciding which pages to refresh first, we will build a **ranked priority queue** from **historical search performance data** (~30,000 pseudonymized content items), predicting **content decay** (measured by trailing 90-day trend direction) at **Precision@50**. A wrong call costs **$150–$300 in wasted editor labor** per page, or missed decay compounds into thousands of lost visits. A plain rule isn't enough because **decay is driven by non-linear interactions across age, impressions, and update recency that fixed thresholds cannot capture**. We will claim only **observed, directional, decision-support** results.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.